# Kaggle Full 03 - Evaluation and Threshold Search (Self-Contained)

This notebook evaluates a trained checkpoint and chooses threshold by ACER.


In [ ]:
# !pip install -q torch torchvision opencv-python pandas numpy matplotlib

In [ ]:
from pathlib import Path
from dataclasses import dataclass
import json

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset

In [ ]:
class ManifestDataset(Dataset):
    LEGACY_PREP_ROOT = Path('/kaggle/working/celeba_spoof_prepared_full')

    def __init__(self, manifest_path: str, image_size: int = 80):
        self.df = pd.read_csv(manifest_path)
        self.image_size = image_size
        self.manifest_path = Path(manifest_path)
        self.prepared_root = self.manifest_path.parent.parent

    def __len__(self):
        return len(self.df)

    def _resolve_image_path(self, raw_path: str) -> Path:
        text = str(raw_path)
        path = Path(text)

        if path.exists():
            return path

        legacy_prefix = str(self.LEGACY_PREP_ROOT) + '/'
        if text.startswith(legacy_prefix):
            suffix = text[len(legacy_prefix):]
            candidate = self.prepared_root / suffix
            if candidate.exists():
                return candidate

        marker = 'crops_80x80/'
        if marker in text:
            suffix = text.split(marker, 1)[1]
            candidate = self.prepared_root / 'crops_80x80' / suffix
            if candidate.exists():
                return candidate

        if not path.is_absolute():
            candidate = self.prepared_root / path
            if candidate.exists():
                return candidate

        return path

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        resolved_path = self._resolve_image_path(row.image_path)
        image = cv2.imread(str(resolved_path))
        if image is None:
            raise RuntimeError(f'Could not load image: {row.image_path} (resolved: {resolved_path})')

        image = cv2.resize(image, (self.image_size, self.image_size))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = image.astype(np.float32) / 255.0
        image = np.transpose(image, (2, 0, 1))
        return torch.from_numpy(image), int(row.label), str(resolved_path)


class SmallFASNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, 1, 1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, 1, 1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, 1, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(0.2),
            nn.Linear(128, 2),
        )

    def forward(self, x):
        return self.net(x)


@dataclass
class EvalMetrics:
    tp: int
    fp: int
    tn: int
    fn: int

    @property
    def accuracy(self):
        total = self.tp + self.fp + self.tn + self.fn
        return (self.tp + self.tn) / total if total else 0.0

    @property
    def apcer(self):
        d = self.fp + self.tn
        return self.fp / d if d else 0.0

    @property
    def bpcer(self):
        d = self.tp + self.fn
        return self.fn / d if d else 0.0

    @property
    def acer(self):
        return (self.apcer + self.bpcer) / 2.0


def evaluate_binary(scores, labels, threshold):
    tp = fp = tn = fn = 0
    for s, y in zip(scores, labels):
        pred_live = s >= threshold
        is_live = y == 1
        if pred_live and is_live:
            tp += 1
        elif pred_live and not is_live:
            fp += 1
        elif (not pred_live) and (not is_live):
            tn += 1
        else:
            fn += 1
    return EvalMetrics(tp, fp, tn, fn)


In [ ]:
MANIFEST_TEST = Path('/kaggle/input/datasets/doraemongwa/celeba-spoof-prepared-full/celeba_spoof_prepared_full/manifests/test.csv')
CHECKPOINT_CANDIDATES = [
    Path('/kaggle/working/celeba_spoof_training_full/best_model.pt'),
    Path('/kaggle/input/results/celeba_spoof_training_full/best_model.pt'),
]
CHECKPOINT = next((p for p in CHECKPOINT_CANDIDATES if p.exists()), CHECKPOINT_CANDIDATES[0])
OUT_ROOT = Path('/kaggle/working/celeba_spoof_eval_full')
OUT_ROOT.mkdir(parents=True, exist_ok=True)

print('manifest exists:', MANIFEST_TEST.exists())
print('checkpoint path:', CHECKPOINT)
print('checkpoint exists:', CHECKPOINT.exists())

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

payload = torch.load(CHECKPOINT, map_location='cpu')
image_size = int(payload.get('image_size', 80))

model = SmallFASNet().to(device)
model.load_state_dict(payload['state_dict'])
model.eval()

ds = ManifestDataset(str(MANIFEST_TEST), image_size=image_size)

rows = []
with torch.no_grad():
    for i in range(len(ds)):
        x, y, p = ds[i]
        logits = model(x.unsqueeze(0).to(device))
        score = float(torch.softmax(logits, dim=1)[0, 1].item())
        rows.append({'image_path': p, 'label': y, 'live_score': score})

pred_df = pd.DataFrame(rows)
print('rows:', len(pred_df))
display(pred_df.head(5))

In [ ]:
thresholds = np.linspace(0.05, 0.95, 19)
metrics_rows = []
for t in thresholds:
    m = evaluate_binary(pred_df['live_score'].tolist(), pred_df['label'].tolist(), float(t))
    metrics_rows.append(
        {
            'threshold': float(t),
            'accuracy': m.accuracy,
            'apcer': m.apcer,
            'bpcer': m.bpcer,
            'acer': m.acer,
            'tp': m.tp,
            'fp': m.fp,
            'tn': m.tn,
            'fn': m.fn,
        }
    )

metrics_df = pd.DataFrame(metrics_rows).sort_values('acer').reset_index(drop=True)
best = metrics_df.iloc[0].to_dict()
print('best:', best)
display(metrics_df.head(10))

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(metrics_df['threshold'], metrics_df['acer'], marker='o', label='ACER')
plt.plot(metrics_df['threshold'], metrics_df['accuracy'], marker='s', label='Accuracy')
plt.xlabel('Threshold')
plt.ylabel('Metric')
plt.title('Threshold sweep')
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
pred_path = OUT_ROOT / 'predictions.csv'
metrics_path = OUT_ROOT / 'threshold_metrics.csv'
best_path = OUT_ROOT / 'best_threshold.json'

pred_df.to_csv(pred_path, index=False)
metrics_df.to_csv(metrics_path, index=False)
best_path.write_text(json.dumps(best, indent=2))

print('saved predictions:', pred_path)
print('saved threshold metrics:', metrics_path)
print('saved best threshold:', best_path)